# Lab 2: Feature Engineering + Improved Baseline
## IIT414W · Unit II

**Integrantes:** David Hernandez y Ariel Van Kilsdonk

**Objetivo:** superar los baselines de Lab 1 usando features pre-race y un modelo simple (Logistic Regression), manteniendo la misma validacion temporal (Train 2022, Validation 2023, Test 2024 sellado).

**Configuracion obligatoria:** `RANDOM_SEED = 414`

**Target:** `top10` (1 si termina Top-10, 0 en caso contrario).

In [1]:
# Reproducibility header
import sys, random
import warnings
import numpy as np
import pandas as pd
import requests

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

RANDOM_SEED = 414
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

print(f'Python: {sys.version.split()[0]}')
print(f'NumPy : {np.__version__}')
print(f'pandas: {pd.__version__}')
print(f'Seed  : {RANDOM_SEED}')

Python: 3.12.3
NumPy : 2.1.0
pandas: 2.3.3
Seed  : 414


## 1) Data Loading (Jolpica API)

Cargamos 2019-2024 para construir historial de features, pero evaluamos con el mismo split de Lab 1 (Train 2022, Validation 2023, Test 2024).

In [2]:
SEASONS = [2019, 2020, 2021, 2022, 2023, 2024]

all_rows = []
for year in SEASONS:
    url = f'https://api.jolpi.ca/ergast/f1/{year}/results.json?limit=1000'
    payload = requests.get(url, timeout=30).json()

    for race in payload['MRData']['RaceTable']['Races']:
        for result in race.get('Results', []):
            all_rows.append({
                'season': year,
                'round': int(race['round']),
                'race_name': race['raceName'],
                'circuit_id': race['Circuit']['circuitId'],
                'date': race['date'],
                'driver_id': result['Driver']['driverId'],
                'driver_name': f"{result['Driver']['givenName']} {result['Driver']['familyName']}",
                'constructor': result['Constructor']['name'],
                'constructor_id': result['Constructor']['constructorId'],
                'grid': int(result['grid']),
                'position_str': result.get('position', 'R'),
                'position_order': int(result.get('positionOrder', 99)),
                'points': float(result.get('points', 0.0)),
                'status': result.get('status', 'Unknown'),
            })

    print(f'{year}: {len(payload["MRData"]["RaceTable"]["Races"])} races loaded')

df = pd.DataFrame(all_rows)
df['date'] = pd.to_datetime(df['date'])
df['position'] = pd.to_numeric(df['position_str'], errors='coerce')
df['top10'] = (df['position'] <= 10).astype(int)
df.loc[df['position'].isna(), 'top10'] = 0

print(f'Loaded rows: {len(df):,}')
print(f'Unique races: {df.groupby(["season", "round"]).ngroups}')
print(f'Top-10 rate : {df["top10"].mean():.1%}')

2019: 5 races loaded
2020: 5 races loaded
2021: 5 races loaded
2022: 5 races loaded
2023: 5 races loaded
2024: 6 races loaded
Loaded rows: 600
Unique races: 31
Top-10 rate : 50.2%


In [3]:
# Temporal split: identical to Lab 1
df_train = df[df['season'] == 2022].copy()
df_val = df[df['season'] == 2023].copy()
df_test = df[df['season'] == 2024].copy()  # sealed

print('Train (2022):', len(df_train))
print('Val   (2023):', len(df_val))
print('Test  (2024):', len(df_test), '[SEALED]')

Train (2022): 100
Val   (2023): 100
Test  (2024): 100 [SEALED]


## 2) Lab 1 Baselines (same validation and metrics)

Baselines incluidos:
- Majority class (DummyClassifier)
- Domain heuristic: `grid <= 10`

In [4]:
def binary_metrics(y_true, y_pred, y_prob=None):
    out = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
    }
    out['ROC-AUC'] = roc_auc_score(y_true, y_prob) if y_prob is not None else np.nan
    return out

# Majority class baseline
X_train_base = df_train[['grid']]
X_val_base = df_val[['grid']]
y_train_base = df_train['top10']
y_val_base = df_val['top10']

clf_majority = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
clf_majority.fit(X_train_base, y_train_base)
pred_majority = clf_majority.predict(X_val_base)

# Domain heuristic baseline
pred_heuristic = (df_val['grid'] <= 10).astype(int)

majority_metrics = binary_metrics(y_val_base, pred_majority)
heuristic_metrics = binary_metrics(y_val_base, pred_heuristic)

print('Majority class metrics:', majority_metrics)
print('Domain heuristic metrics:', heuristic_metrics)

Majority class metrics: {'Accuracy': 0.5, 'Precision': np.float64(0.0), 'Recall': np.float64(0.0), 'F1': np.float64(0.0), 'ROC-AUC': nan}
Domain heuristic metrics: {'Accuracy': 0.74, 'Precision': np.float64(0.74), 'Recall': np.float64(0.74), 'F1': np.float64(0.74), 'ROC-AUC': nan}


## 3) Feature Engineering (3+ features, pre-race only)

### Feature 1: `prev_position` (lag)
Usa la posicion final de la carrera anterior por piloto. Resume momentum reciente sin mirar la carrera objetivo.

### Feature 2: `avg_position_last3` (rolling)
Promedio de posicion en las ultimas 3 carreras por piloto, con `shift(1)` para evitar leakage de la fila actual.

### Feature 3: `driver_circuit_avg_prev` (interaction historica)
Promedio historico previo del piloto en ese circuito. Captura afinidad piloto-circuito pre-race.

### Feature 4: `constructor_tier_prev_season` (categorical encoding)
Tier `top/mid/back` segun puntos de la temporada anterior del constructor. Resume competitividad estructural del equipo antes de iniciar la temporada actual.

In [5]:
feat_df = df.sort_values(['driver_id', 'season', 'round']).copy()

# 1) Lag feature
feat_df['prev_position'] = feat_df.groupby('driver_id')['position'].shift(1)

# 2) Rolling aggregate feature
feat_df['avg_position_last3'] = (
    feat_df.groupby('driver_id')['position']
    .transform(lambda x: x.rolling(3, min_periods=1).mean().shift(1))
)

# 3) Interaction feature (driver x circuit historical performance)
grp = feat_df.groupby(['driver_id', 'circuit_id'])['position']
cum_sum = grp.cumsum() - feat_df['position']
cum_cnt = grp.cumcount()
feat_df['driver_circuit_avg_prev'] = cum_sum / cum_cnt.replace(0, np.nan)

# 4) Constructor tier from previous season points
season_constructor_points = (
    feat_df.groupby(['season', 'constructor_id'], as_index=False)['points']
    .sum()
    .rename(columns={'points': 'season_points'})
)
season_constructor_points['season'] = season_constructor_points['season'] + 1
season_constructor_points = season_constructor_points.rename(columns={'season_points': 'prev_season_points'})

feat_df = feat_df.merge(season_constructor_points, on=['season', 'constructor_id'], how='left')
rank_pct = feat_df.groupby('season')['prev_season_points'].rank(pct=True)
feat_df['constructor_tier_prev_season'] = np.where(
    rank_pct >= (2/3), 'top', np.where(rank_pct >= (1/3), 'mid', 'back')
)
feat_df['constructor_tier_prev_season'] = feat_df['constructor_tier_prev_season'].fillna('mid')

feature_cols = [
    'grid',
    'prev_position',
    'avg_position_last3',
    'driver_circuit_avg_prev',
    'constructor_tier_prev_season',
]

feat_df[feature_cols + ['top10']].head()

,grid,prev_position,avg_position_last3,driver_circuit_avg_prev,constructor_tier_prev_season,top10
0,13,NaN,NaN,NaN,back,0
1,12,14.0,14.0,NaN,back,1
2,0,9.0,11.5,NaN,back,1
3,11,10.0,11.0,NaN,back,0
4,11,11.0,10.0,NaN,back,0


## 4) Leakage Guard Checklist (10 items)

| # | Check | Resultado |
|---|-------|-----------|
| 1 | Solo info pre-race por fila objetivo | PASS |
| 2 | Split temporal igual a Lab 1 | PASS |
| 3 | No uso directo del target (`top10`) en features | PASS |
| 4 | No uso de `position` de la misma carrera objetivo | PASS |
| 5 | Features temporales con `shift(1)` | PASS |
| 6 | Agregaciones historicas por grupo, no futuras | PASS |
| 7 | Encoding de constructor con temporada previa | PASS |
| 8 | Test 2024 no usado para decisiones del modelo | PASS |
| 9 | `RANDOM_SEED = 414` en random_state | PASS |
| 10 | Notebook reproducible top-to-bottom | PASS (verificar al ejecutar completo) |

In [6]:
# Temporal split for Lab 2 model (same boundaries)
lab2_train = feat_df[feat_df['season'] == 2022].copy()
lab2_val = feat_df[feat_df['season'] == 2023].copy()

# Impute temporal features to avoid losing all rows early in a season
for c in ['prev_position', 'avg_position_last3', 'driver_circuit_avg_prev']:
    train_median = lab2_train[c].median()
    if pd.isna(train_median):
        train_median = lab2_train['position'].median()
    lab2_train[c] = lab2_train[c].fillna(train_median)
    lab2_val[c] = lab2_val[c].fillna(train_median)

X_train_lab2 = lab2_train[feature_cols].copy()
X_val_lab2 = lab2_val[feature_cols].copy()
y_train_lab2 = lab2_train['top10'].astype(int)
y_val_lab2 = lab2_val['top10'].astype(int)

X_train_lab2 = pd.get_dummies(X_train_lab2, columns=['constructor_tier_prev_season'], drop_first=False)
X_val_lab2 = pd.get_dummies(X_val_lab2, columns=['constructor_tier_prev_season'], drop_first=False)
X_train_lab2, X_val_lab2 = X_train_lab2.align(X_val_lab2, join='left', axis=1, fill_value=0)

model = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
model.fit(X_train_lab2, y_train_lab2)

pred_lab2 = model.predict(X_val_lab2)
prob_lab2 = model.predict_proba(X_val_lab2)[:, 1]

lab2_metrics = binary_metrics(y_val_lab2, pred_lab2, y_prob=prob_lab2)
print('Lab 2 model metrics:', lab2_metrics)

Lab 2 model metrics: {'Accuracy': 0.73, 'Precision': np.float64(0.7804878048780488), 'Recall': np.float64(0.64), 'F1': np.float64(0.7032967032967034), 'ROC-AUC': np.float64(0.7796000000000001)}


In [7]:
comparison_df = pd.DataFrame([
    {'Model / Baseline': 'Majority class (Lab 1)', **majority_metrics},
    {'Model / Baseline': 'Domain heuristic (Lab 1)', **heuristic_metrics},
    {'Model / Baseline': 'Prior-period (if done)', 'Accuracy': np.nan, 'Precision': np.nan, 'Recall': np.nan, 'F1': np.nan, 'ROC-AUC': np.nan},
    {'Model / Baseline': 'Lab 2 model (LogReg)', **lab2_metrics},
])

for c in ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']:
    comparison_df[c] = comparison_df[c].round(4)

comparison_df

,Model / Baseline,Accuracy,Precision,Recall,F1,ROC-AUC
0,Majority class (Lab 1),0.50,0.0000,0.00,0.0000,NaN
1,Domain heuristic (Lab 1),0.74,0.7400,0.74,0.7400,NaN
2,Prior-period (if done),NaN,NaN,NaN,NaN,NaN
3,Lab 2 model (LogReg),0.73,0.7805,0.64,0.7033,0.7796


## 5) Error Analysis (Top-3 failure modes)

Se identifican modos de falla concretos por carrera, piloto y tipo de error (FP/FN).

In [9]:
err_df = lab2_val[['season', 'round', 'race_name', 'driver_id', 'driver_name', 'constructor', 'grid', 'position', 'top10']].copy()
err_df['y_pred'] = pred_lab2
err_df['y_prob'] = prob_lab2
err_df['is_error'] = (err_df['top10'] != err_df['y_pred']).astype(int)

race_error = (
    err_df.groupby(['season', 'round', 'race_name'], as_index=False)['is_error']
    .mean()
    .sort_values('is_error', ascending=False)
)

driver_error = (
    err_df.groupby(['driver_id', 'driver_name'])['is_error']
    .agg(['mean', 'count'])
    .reset_index()
)
driver_error.columns = ['driver_id', 'driver_name', 'error_rate', 'n']
driver_error = driver_error[driver_error['n'] >= 3].sort_values('error_rate', ascending=False)

fp = err_df[(err_df['top10'] == 0) & (err_df['y_pred'] == 1)].copy()
fn = err_df[(err_df['top10'] == 1) & (err_df['y_pred'] == 0)].copy()

print('Top 3 races by error rate:')
display(race_error.head(3))

print('Top 3 drivers by error rate (n>=3):')
display(driver_error.head(3))

print('Error type counts:')
print(pd.Series({'False Positives': len(fp), 'False Negatives': len(fn)}))

print('\nFailure mode guide (adapt with your run output):')
print('1) Carrera con alta tasa de error: posible caoticidad (safety car, incidentes).')
print('2) Pilotos con alta varianza: falta feature de consistencia (rolling std).')
print('3) FP/FN dominantes: calibrar threshold o agregar interacciones constructor-circuito.')

Top 3 races by error rate:


,season,round,race_name,is_error
0,2023,1,Bahrain Grand Prix,0.45
2,2023,3,Australian Grand Prix,0.45
1,2023,2,Saudi Arabian Grand Prix,0.25


Top 3 drivers by error rate (n>=3):


,driver_id,driver_name,error_rate,n
10,norris,Lando Norris,0.6,5
0,albon,Alexander Albon,0.4,5
4,gasly,Pierre Gasly,0.4,5


Error type counts:
False Positives     9
False Negatives    18
dtype: int64

Failure mode guide (adapt with your run output):
1) Carrera con alta tasa de error: posible caoticidad (safety car, incidentes).
2) Pilotos con alta varianza: falta feature de consistencia (rolling std).
3) FP/FN dominantes: calibrar threshold o agregar interacciones constructor-circuito.


In [10]:
print(comparison_df.to_string(index=False))

print('\nPrimary metric (same as Lab 1): Accuracy')
print('If your Lab 1 primary metric was F1, update this label and discussion accordingly.')

        Model / Baseline  Accuracy  Precision  Recall     F1  ROC-AUC
  Majority class (Lab 1)      0.50     0.0000    0.00 0.0000      NaN
Domain heuristic (Lab 1)      0.74     0.7400    0.74 0.7400      NaN
  Prior-period (if done)       NaN        NaN     NaN    NaN      NaN
    Lab 2 model (LogReg)      0.73     0.7805    0.64 0.7033   0.7796

Primary metric (same as Lab 1): Accuracy
If your Lab 1 primary metric was F1, update this label and discussion accordingly.


## 6) Error Analysis — Top 3 Failure Modes

### Failure Mode 1: Carreras de inicio de temporada (Bahrain GP, Australian GP)
- **Error rate:** 45%
- **Hipótesis:** Las features históricas (`prev_position`, `avg_position_last3`, `driver_circuit_avg_prev`) tienen muchos NaN o valores imputados al inicio de la temporada, reduciendo su poder predictivo.
- **Siguiente paso:** Crear feature `races_in_season` para ponderar la confianza en features históricas, o usar datos de pretemporada/testing.

### Failure Mode 2: Pilotos de mid-field con alta varianza (Norris, Albon, Gasly)
- **Error rate:** 40-60%
- **Hipótesis:** Pilotos en equipos de mitad de tabla tienen resultados inconsistentes (a veces Top-10, a veces no), y las features actuales no capturan esta varianza.
- **Siguiente paso:** Agregar feature `rolling_std_position` para medir consistencia del piloto, y `constructor_reliability` para capturar DNFs mecánicos.

### Failure Mode 3: Dominancia de False Negatives (18 FN vs 9 FP)
- **Hipótesis:** El modelo es demasiado conservador — predice "No Top-10" cuando debería predecir "Top-10". El threshold de 0.5 puede no ser óptimo.
- **Siguiente paso:** (1) Calibrar el threshold de probabilidad, (2) Agregar features que capturen oportunidades de avance (pit stops más rápidos, degradación de neumáticos del circuito).

---

## Conclusión

El modelo de Logistic Regression con las 4 features engineered **no superó el domain heuristic** (73% vs 74% accuracy). Sin embargo:

1. **Mayor precision (78% vs 74%):** Cuando el modelo predice Top-10, es más confiable.
2. **ROC-AUC de 0.78:** El modelo tiene capacidad discriminativa razonable.
3. **Las features capturan información redundante:** La posición de grilla ya incorpora mucha de la información que las features históricas intentan capturar.

**Para Lab 3:** Explorar features menos correlacionadas con grid position, como condiciones meteorológicas, fiabilidad mecánica del constructor, y rendimiento específico en tipos de circuito (street vs permanent).